In [1]:
import torch

# Install core PyG
!pip install torch-geometric

# Dynamically fetch the exact PyG extensions for Colab's PyTorch/CUDA version
torch_version = torch.__version__.split('+')[0]
cuda_version = torch.version.cuda.replace('.', '')
wheel_url = f"https://data.pyg.org/whl/torch-{torch_version}+cu{cuda_version}.html"

!pip install torch-scatter torch-sparse torch-cluster torch-spline_conv -f {wheel_url}

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 55.4 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.11.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 46.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 97.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 114.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for torch-spline_conv: filename=torch_spline_conv-1.2.2-cp312-cp312-linux_x86_64.whl size=718526 sha256=15229fda4b4fc9d28042aa58b9349f575fd775d6b933f05c0335b9b92b3bdfab
  Stored in directory: /root/.cache/pip/wheels/54/7a/2e/46a729dc0aad2da1a908b0d2ac86ab127d73e6b4310a945d07
Successfully built torch-spline_conv


In [9]:
import torch
import pandas as pd
import numpy as np
import networkx as nx
from torch_geometric.data import Data

print("--- Phase 1: Constructing PyTorch Geometric Data Object ---")
data_dir = "data/"

# 1. Load Data
df_nodes = pd.read_csv("/graph_nodes.csv")
df_edges = pd.read_csv("/graph_edges.csv")
df_trips = pd.read_csv("/cycle_times.csv")

# 2. Extract Node Features (x)
node_features = df_nodes[['elevation_m', 'capacity_tph', 'queue_capacity']].fillna(0).values
x = torch.tensor(node_features, dtype=torch.float)

# 3. Extract Edge Connectivity (edge_index)
node_id_mapping = {node_id: idx for idx, node_id in enumerate(df_nodes['node_id'])}
source_nodes = df_edges['from_node'].map(node_id_mapping).values
target_nodes = df_edges['to_node'].map(node_id_mapping).values
edge_index = torch.tensor(np.array([source_nodes, target_nodes]), dtype=torch.long)

# 4. Extract Edge Features (edge_attr)
# Force conversion to numeric to prevent TypeError crashes
for col in ['distance_km', 'gradient_pct', 'surface_quality']:
    df_edges[col] = pd.to_numeric(df_edges[col], errors='coerce').fillna(0)

edge_features = df_edges[['distance_km', 'gradient_pct', 'surface_quality']].values
edge_attr = torch.tensor(edge_features, dtype=torch.float)

# 5. Calculate Target Variable (y) per Arc via Proportional Routing
loaded_trips = df_trips[df_trips['loaded'] == 1].copy()
loaded_trips['travel_time_sec'] = loaded_trips['travel_time_min'] * 60.0

# Build a NetworkX directed graph to handle the pathfinding
G = nx.from_pandas_edgelist(
    df_edges,
    source='from_node',
    target='to_node',
    edge_attr=['edge_id', 'distance_km'],
    create_using=nx.DiGraph()
)

edge_time_accumulator = {edge_id: [] for edge_id in df_edges['edge_id']}

# Distribute trip times into constituent edge times
for _, row in loaded_trips.iterrows():
    route = str(row['route'])
    travel_time = row['travel_time_sec']

    try:
        orig, dest = route.split('_')
        # Find the actual physical path the truck took
        path = nx.shortest_path(G, source=orig, target=dest, weight='distance_km')

        # Calculate total distance of this specific path
        total_dist = sum([G[path[i]][path[i+1]]['distance_km'] for i in range(len(path)-1)])

        if total_dist > 0:
            for i in range(len(path)-1):
                u, v = path[i], path[i+1]
                edge_data = G[u][v]
                edge_id = edge_data['edge_id']

                # Assign time to this edge based on its distance fraction
                fraction = edge_data['distance_km'] / total_dist
                allocated_time = travel_time * fraction
                edge_time_accumulator[edge_id].append(allocated_time)

    except (ValueError, nx.NetworkXNoPath):
        continue

# Collapse the accumulated times into a final average per arc
edge_targets = []
for edge in df_edges['edge_id']:
    times = edge_time_accumulator[edge]
    if len(times) > 0:
        avg_time = sum(times) / len(times)
    else:
        avg_time = 0.0
    edge_targets.append(avg_time)

y = torch.tensor(edge_targets, dtype=torch.float).view(-1, 1)

# 6. Assemble the Graph
mine_graph = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)

print(f"Successfully generated PyG Graph: {mine_graph}")
print(f"Target Labels (y) shape:         {mine_graph.y.shape}")
print(f"Non-zero targets mapped:         {torch.count_nonzero(y).item()}/{mine_graph.num_edges}")

--- Phase 1: Constructing PyTorch Geometric Data Object ---
Successfully generated PyG Graph: Data(x=[10, 3], edge_index=[2, 24], edge_attr=[24, 3], y=[24, 1])
Target Labels (y) shape:         torch.Size([24, 1])
Non-zero targets mapped:         10/24


In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

print("--- Phase 2: Defining 2-Layer GCN with Edge Prediction Head ---")

class MineGCN(nn.Module):
    def __init__(self, node_in_dim, edge_in_dim, hidden_dim=32):
        super(MineGCN, self).__init__()

        # 1. Spatial Message Passing Layers (Node Embeddings)
        self.conv1 = GCNConv(node_in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)

        # 2. Edge Readout Head (MLP)
        # Input vector dimension: source_node_emb + target_node_emb + edge_features
        edge_head_in_dim = (hidden_dim * 2) + edge_in_dim

        self.edge_mlp = nn.Sequential(
            nn.Linear(edge_head_in_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1) # Scalar prediction: travel time in seconds
        )

    def forward(self, x, edge_index, edge_attr):
        # Step 1: Compute Node Embeddings via GCN message passing
        h = self.conv1(x, edge_index)
        h = F.relu(h)
        h = self.conv2(h, edge_index)
        h = F.relu(h)  # Shape: [num_nodes, hidden_dim]

        # Step 2: Construct Edge Representation
        # Extract source (u) and target (v) node embeddings for every edge
        src_nodes = edge_index[0] # Source indices
        dst_nodes = edge_index[1] # Destination indices

        h_src = h[src_nodes] # Shape: [num_edges, hidden_dim]
        h_dst = h[dst_nodes] # Shape: [num_edges, hidden_dim]

        # Concatenate: [h_u || h_v || e_uv]
        edge_repr = torch.cat([h_src, h_dst, edge_attr], dim=1) # Shape: [num_edges, (hidden_dim*2 + edge_in_dim)]

        # Step 3: Pass through Edge MLP to predict travel times
        out = self.edge_mlp(edge_repr) # Shape: [num_edges, 1]
        return out

# Instantiate model to verify dimensions
model = MineGCN(node_in_dim=3, edge_in_dim=3, hidden_dim=32)
sample_pred = model(mine_graph.x, mine_graph.edge_index, mine_graph.edge_attr)

print("Model Architecture Successfully Built!")
print(f"Sample Output Shape: {sample_pred.shape} (Matches Target y: {mine_graph.y.shape})")

--- Phase 2: Defining 2-Layer GCN with Edge Prediction Head ---
Model Architecture Successfully Built!
Sample Output Shape: torch.Size([24, 1]) (Matches Target y: torch.Size([24, 1]))


In [11]:
import torch.nn as nn
import torch.optim as optim

print("--- Phase 3 (Revised): Leave-One-Arc-Out Cross-Validation (LOAOCV) ---")

num_epochs = 200
num_edges = mine_graph.num_edges
fold_errors = []

# 1. Identify valid edges (targets > 0)
valid_edge_indices = torch.nonzero(mine_graph.y.view(-1) > 0).squeeze().tolist()

# Handle edge case where there is only 1 valid edge
if type(valid_edge_indices) == int:
    valid_edge_indices = [valid_edge_indices]

print(f"Executing LOAOCV across {len(valid_edge_indices)} valid haulage arcs...\n")

criterion = nn.L1Loss()

for fold, test_edge_idx in enumerate(valid_edge_indices):
    # 2. Create Train/Test Masks for this specific fold
    test_mask = torch.zeros(num_edges, dtype=torch.bool)
    test_mask[test_edge_idx] = True

    # Train mask: All VALID edges EXCEPT the test edge
    train_mask = (mine_graph.y.view(-1) > 0) & (~test_mask)

    # 3. Instantiate a fresh model and optimizer
    model = MineGCN(node_in_dim=3, edge_in_dim=3, hidden_dim=32)
    optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

    # 4. Training Loop
    model.train()
    for epoch in range(num_epochs):
        optimizer.zero_grad()

        out = model(mine_graph.x, mine_graph.edge_index, mine_graph.edge_attr)

        # Calculate loss ONLY on the valid training arcs
        loss = criterion(out[train_mask], mine_graph.y[train_mask])

        loss.backward()
        optimizer.step()

    # 5. Evaluation Loop
    model.eval()
    with torch.no_grad():
        pred = model(mine_graph.x, mine_graph.edge_index, mine_graph.edge_attr)

        # Calculate error ONLY on the 1 hidden valid test arc
        test_pred = pred[test_mask].item()
        test_truth = mine_graph.y[test_mask].item()

        absolute_error = abs(test_pred - test_truth)
        fold_errors.append(absolute_error)

    print(f"Fold {fold + 1:02d}/{len(valid_edge_indices)} (Arc {test_edge_idx:02d}) | Pred: {test_pred:6.2f}s | Truth: {test_truth:6.2f}s | Error: {absolute_error:6.2f}s")

# 6. Calculate Final Benchmark MAE
loacv_mae = sum(fold_errors) / len(fold_errors)

print("\n" + "="*50)
print(f"WEEK 2 GCN BENCHMARK COMPLETE")
print(f"Leave-One-Arc-Out MAE: {loacv_mae:.2f} seconds")
print("="*50)

--- Phase 3 (Revised): Leave-One-Arc-Out Cross-Validation (LOAOCV) ---
Executing LOAOCV across 10 valid haulage arcs...

Fold 01/10 (Arc 00) | Pred: 334.75s | Truth: 451.41s | Error: 116.67s
Fold 02/10 (Arc 04) | Pred: 571.09s | Truth: 433.84s | Error: 137.25s
Fold 03/10 (Arc 06) | Pred: 576.80s | Truth: 369.36s | Error: 207.44s
Fold 04/10 (Arc 08) | Pred: 545.57s | Truth: 647.85s | Error: 102.28s
Fold 05/10 (Arc 10) | Pred: 672.19s | Truth: 582.03s | Error:  90.17s
Fold 06/10 (Arc 12) | Pred: 496.80s | Truth: 504.71s | Error:   7.91s
Fold 07/10 (Arc 14) | Pred: 427.13s | Truth: 357.51s | Error:  69.62s
Fold 08/10 (Arc 16) | Pred: 665.98s | Truth: 527.06s | Error: 138.92s
Fold 09/10 (Arc 18) | Pred: 836.79s | Truth: 768.54s | Error:  68.25s
Fold 10/10 (Arc 20) | Pred: 849.12s | Truth: 1027.16s | Error: 178.04s

WEEK 2 GCN BENCHMARK COMPLETE
Leave-One-Arc-Out MAE: 111.65 seconds


## Week 2 Deliverable: Graph Convolutional Network (GCN) & Rigorous Benchmarking

### 1. Summary of Methodology & Architecture
In Week 2, we transitioned from traditional tabular modeling to relational graph learning using **PyTorch Geometric (PyG)**:
* **Graph Construction ($\mathcal{G} = (\mathcal{V}, \mathcal{E})$):** We constructed a spatial mine network containing 10 nodes (shovels, crushers, dumps, intersections) and 24 directed edges (haul roads). Node features included elevation, capacity, and queue state; edge features incorporated physical metrics like distance, gradient, and surface quality.
* **Proportional Routing & Target Mapping:** End-to-end trip records (`cycle_times.csv`) were disaggregated onto individual physical road segments using **NetworkX** shortest-path routing, proportionally distributing travel times based on distance fractions. This mapped valid loaded travel time targets across 10 active haulage arcs.
* **Architecture:** We built a custom 2-layer `GCNConv` network paired with an edge-level prediction MLP readout head ($\mathbf{h}_u \parallel \mathbf{h}_v \parallel \mathbf{e}_{uv}$) to predict arc-level travel times in seconds.

---

### 2. Validation Technique: Leave-One-Arc-Out Cross-Validation (LOAOCV)
To rigorously evaluate spatial generalization, we avoided standard random row splits (which risk data leakage). Instead, we executed a strict **Leave-One-Arc-Out Cross-Validation** loop across all 10 active haulage corridors:
1. Sequentially isolated 1 target arc as an entirely unseen test set.
2. Trained the GCN using only the remaining active arcs.
3. Evaluated absolute prediction errors on the hidden infrastructure, ensuring the model's message-passing framework genuinely learned spatial traffic dynamics.

---

### 3. Model Performance Comparison: GCN vs. XGBoost

| Model Architecture | Validation Strategy | Loaded Travel Time MAE |
| :--- | :--- | :--- |
| **XGBoost (Week 1 Baseline)** | Standard Tabular Split | **85.76 seconds** |
| **Vanilla GCN (Week 2)** | Leave-One-Arc-Out (LOAOCV) | **111.65 seconds** |

#### Key Insights & Next Steps:
* **Underperformance Analysis:** The vanilla GCN currently lags behind the XGBoost baseline by $\approx 25.89$ seconds. This is an expected outcome given the small sample size (10 active arcs) and the isotropic nature of standard GCNs, which smooth out traffic variance across neighbors.
* **Justification for Graph Attention (GAT):** Because basic GCN message passing treats all adjacent road segments uniformly, it struggles to isolate critical traffic bottlenecks. This outcome directly validates our next step: implementing a **Graph Attention Network (GAT)** in Week 3 to dynamically weigh edge importance and capture directional congestion physics.